### War & Econmic Impact
##### `Project Overview`

**Datasest:** [War_economic_impact_dataset.csv](https://www.kaggle.com/datasets/likithagedipudi/war-economic-and-livelihood-impact-dataset) - 100,000 rows, 28 columns
**Regoions:** Africa, East Asia, Europe, Middle East, South Asia
**Conflict Types:** Asymmetric War, Civil War, Interstate War, Interstate/Counter-insuergency, World War
**Years:** 1939 - 2026
**Null Values:** Zero

**Research Questions:** 
- *Regression* : Which economic indicators predict the total financial scale of a conflict?
- *Classification* : Can a conflict's economic damage profile predict whether it is ongoing or resolved?

**Feature Engineering:**
- War_profiteering_documented - > Binary Encoded (i.e Yes/No => 1/0)
- Black_market_activity_level - > Ordinal encoded (i.e Low-Dominant -> 1-4)
- Duration_in_years - > End_year - Start_year
- Informal_Economy_Size_Spike_% - > wartime level - pre-war level
- Reconstruction_Cost_Ratio_USD - > Reconstruction cost/ war cost
- Poverty_Rate_Spike_% -> wartime rate - pre-war rate
- Human_Eco_Proxy - > Minmax (poverty households) + Minmax (war cost)

---

##### `Key Visuals`

**Figure 1 - GPD Change by Conflict Type & Region**
- *Why:* Tests whether economic damage is driven by the kind of war or where it occurs. 
- *Findings:* World Wars show the deepest median GPD contractions (~55% decline). The Middle East shows the widest spread across all conflict types, reflecting high within-region diversity.

![alt text](Figures/fig1_gdp_conflict_region.png)

**Figure 2 - Correlation Heatmap**
- *Why:* Identifies multicollinearity and confirms which features co-move with targets before modelling.
- *Findings* Inflation and Currency Devaluation are strongly correlated (r ~ 0.85). GDP Change correlates negatively with Unemployment Spike and Poverty Spike as expected.

![alt text](Figures/fig2_correlation_heatmap.png)

**Figure 3 - Poverty Rate Spike: Ongoing vs Resolved (Violin + CDF)**
- *Why:* Tests whether ending a conflict reduces poverty outcomes. Directly motivates theMann-Whitney test and the classification task.
- *Findings:* Historical resolved conflicts (WWII, Korean War) pull the resolved group's upper tail higher (a period confound that requires an era-level control variable).


![alt text](Figures/fig3_poverty_spike_status.png)

**Figure 4 - Informal Economy Growth vs Black Market Activity**
- *Why:* Tests whether black market severity drives informal economy growth, relevant to sanctions policy and post-conflict recovery design.
- *Findings:* Median informal economy spike rises monotonically with black market level. Profiteering concentrates at higher levels, though no statistically significant association was found.

![alt text](Figures/fig4_blackmarket_informal.png)

---

##### `Statistical Results`

All tests are non-parametric *(heavy-tailed distributions, n = 100,000)*. *Effect sizes* are the primary decision criterion. The *p-values* alone are insufficient at this sample size.

| Test | Question | Statistic | P-value | Effect Size | Decision |
|------|----------|-----------|---------|-------------|----------|
| Chi-Square | Black Market Activity implies Profiteering | $X_2$  2.07 | 0.56 | Cramer's V = 0.005 | Fail to reject $H_0$ |
|Krushal-Wallis H | GDP change across conflict types | H = 25,054 | < 0.001 | $eta_2$ = 0.251 | Reject $H_0$ | 
| Mann-Whitney U | Poverty spike: Ongoing > Resolved | U = 1.62B | < 0.001 | r = -0.309 | Reject $H_0$ |


**Direction Reserved:** Rsolved conflicts score higher due to catastrophic historical wars (WWII, Korean War). An era-level control is required to isolate the true effect of conflict resolution.

**Statistical Results Conclusion:** 
- $H_1$: Black market activity level has no meaningful association with documented profiteering. Likely reflects a documentation gap in high-severity, low-institution contexts.
- $H_2$: Conflict type accounts for 25% of variance in GDP decline. World Wars cause qualitatively different damage than asymmetric or civil wars
- $H_3$: The ongoing vs resolved comparison is confounded by era. Historical resolved conflicts had extreme poverty impacts that inflate the resolved group's scores
<br>
---

##### `Model Preformance Table`

**-- Regression: Target = log(Cost_of_War_USD) --**
*Split: 60 / 20 / 20 , Scaling: Standard Scalar, Features: 11 economic + Region + Conflict_Type dummies*
<br>
| Model | Val RMSE | Test RMSE | Test MAE | Test R2 |
|-------|----------|-----------|----------|---------|
| Linear Regression | 0.9600 | 0.9708 | 0.7250 | -0.001 |
| Random Forest | 0.9726 | 0.9859 | 0.7510 | -0.032 |

*Note:* R2 = 0 is a finding, not a failure. Economic damage indicators do not explain war costs. RSME ~0.97 on log scale implies approximately 2.6x error in original dollar terms.

**Top Features (For both models):** Currency_Black_Market_Rate_Gap, Currency_Devaluation, GDP_Change.

| Model | Val Acc | Test Acc | Percision | Recall | F1 | ROC-AUC |
|-------|---------|----------|-----------|--------|----|---------|
| Logistic Regression | 0.8221 | 0.8231 | 0.8243 | 0.8582 | 0.8409 | 0.9362 | 
| Random Forest | 0.8189 | 0.8226 | 0.8325 | 0.8442 | 0.8384 | 0.9366 |

*Note:* AUC = 0.937 for both is a strong genuine performance. Near-identical scores between LR and RF confirm the decision boundary is approximately linear..The tree interactions add nothing. Recall >
Precision (preferred since missing an active conflict costs more than a false alarm).

**Top Predictors:** Region Middle East, Conflict Type World War, GPD Change

---

##### `Interpretation & Conclution`

- *Finding 1:* Conflict type is the dominant structural predictor of economic damage (eta2 = 0.25). Humanitarian responses calibrated by conflict archetype would be better targeted than uniform ones.
- *Finding 2:* War cost cannot be predicted from economic damage (R2 = 0). Financial scale is driven by geopolitical and military factors. Early-warning systems need institutional data, not just
economic indicators.
- *Finding 3:* Conflict status IS predictable from economic footprint (AUC = 0.937, F1 = 0.84). Geography and conflict type carry the most signal; unemployment spike and GDP decline are the strongest purely economic indicators of an active conflict.
- *Finding 4:* Black market type and profiteering documentation are both independent of all economic indicators tested. They both reflect institutional recording capacity more than economic severity.

**Limitations**
- Near-uniform regional distributions suggest possible data augmentation, limiting causal inference.
- Duration leakage required excluding a key structural feature from classification.
- Omitted confounders: governance quality, sanctions, and aid flows are absent but likely dominant.
- Documentation bias is worst in the highest-severity, lowest-institution contexts.

**Next Steps**
1. Add conflict-era variable (Cold War / post-Cold War / 21st century) to control reporting shifts
2. Use Cox proportional hazards to model time-to-resolution as a continuous outcome.
